# 03-4. CatBoost モデル設計 — 2026-03-08

## リクルート飲食店来客数予測コンペティション

**目的**: 02で確定した特徴量セット（48特徴量）を使い、CatBoostモデルを構築し、全モデルと比較する。

### 本ノートブックの構成
1. データ読み込み（中間データから）
2. モデル概要と選定理由
3. ハイパーパラメータ設定
4. 時系列バリデーションによる学習
5. 予測結果の分析
6. 特徴量重要度の分析
7. 全モデル比較とアンサンブル
8. ハイパーパラメータチューニング（Optuna）
9. まとめ

---
## 1. データ読み込み（中間データから）

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

from catboost import CatBoostRegressor, Pool
from sklearn.metrics import mean_squared_error

%matplotlib inline
plt.rcParams['font.family'] = 'MS Gothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['font.size'] = 12
plt.rcParams['figure.facecolor'] = 'white'

SEED = 42
np.random.seed(SEED)
INTERMEDIATE_DIR = Path('./intermediate')

def rmsle(y_true, y_pred):
    y_pred = np.clip(y_pred, 0, None)
    return np.sqrt(mean_squared_error(np.log1p(y_true), np.log1p(y_pred)))

# 02の中間データ読み込み
with open(INTERMEDIATE_DIR / '02_feature_design.pkl', 'rb') as f:
    prev_02 = pickle.load(f)

confirmed = prev_02['confirmed_settings']
print('=== 02 確定設定 ===')
for k, v in confirmed.items():
    print(f'  {k}: {v}')

train_df = prev_02['train_features']
valid_df = prev_02['valid_features']
all_features = prev_02['feature_columns']['all_features']
VALID_START = prev_02['VALID_START']

# 02で確定した前提条件を適用
TRAIN_START = confirmed['best_train_start']
NAN_STRATEGY = confirmed['best_nan_strategy']

# 01 EDAで設計したval_folds（5フォールド時系列CV）を読み込み
val_folds = prev_02['val_folds']
print(f'\n  CV戦略: {len(val_folds)}フォールド時系列CV（01 EDA設計）')
for i, fold in enumerate(val_folds):
    print(f'    Fold {i+1}: val_start={fold["val_start"]}, val_end={fold["val_end"]}')

# TRAIN_STARTフィルタ適用
before_len = len(train_df)
train_df = train_df[train_df['visit_date'] >= TRAIN_START].reset_index(drop=True)
print(f'\nTRAIN_STARTフィルタ適用: {TRAIN_START}')
print(f'  学習データ: {before_len} → {len(train_df)} 行')

# CatBoostはNaNをネイティブに処理するため、best_nan_strategyに従う
# （LightGBMと同様、NaN戦略は02で検証済み）
print(f'  NaN処理戦略: {NAN_STRATEGY}（CatBoostはNaNネイティブ対応）')

# 先行モデルの結果読み込み
with open(INTERMEDIATE_DIR / '03-1_lgbm_results.pkl', 'rb') as f:
    prev_lgbm = pickle.load(f)
with open(INTERMEDIATE_DIR / '03-2_xgb_results.pkl', 'rb') as f:
    prev_xgb = pickle.load(f)
with open(INTERMEDIATE_DIR / '03-3_rf_results.pkl', 'rb') as f:
    prev_rf = pickle.load(f)

print('\n中間データ読み込み完了')
print(f'  学習データ: {train_df.shape} (期間: {TRAIN_START}～)')
print(f'  検証データ: {valid_df.shape}')
print(f'  特徴量数: {len(all_features)}')
print(f'\n=== 02で確定した前提条件 ===')
print(f'  学習データ期間: {confirmed["best_train_period"]} (start={TRAIN_START})')
print(f'  NaN処理: {NAN_STRATEGY}')
print(f'  Rolling構成: {confirmed["best_rolling_config"]}')
print(f'\n参考スコア:')
print(f'  LightGBM     RMSLE: {prev_lgbm["score_single"]:.5f}')
print(f'  XGBoost      RMSLE: {prev_xgb["score_single"]:.5f}')
print(f'  RandomForest RMSLE: {prev_rf["score_single"]:.5f}')

---
## 2. モデル概要と選定理由

### CatBoostの特徴

| 項目 | CatBoost | LightGBM / XGBoost |
|------|----------|--------------------|
| カテゴリ変数処理 | Ordered Target Statistics（リーク防止内蔵） | ラベルエンコーディングが必要 |
| 木の成長方式 | Symmetric Tree（対称木） | Leaf-wise / Level-wise |
| 過学習防止 | Ordered Boosting | 通常のBoosting |
| 欠損値処理 | 自動（最適方向を学習） | 自動 |

### 選定理由
- **Ordered Boosting**により、少ないデータでも過学習しにくい
- **対称木構造**がLightGBM/XGBoostと異なるため、アンサンブルの多様性に貢献
- NaN処理が内蔵されており、前処理が最小限で済む

---
## 3. ハイパーパラメータ設定

| パラメータ | 値 | 根拠 |
|-----------|-----|------|
| `depth` | 8 | 対称木の深さ（ノード数 = 2^8 = 256） |
| `learning_rate` | 0.02 | 他モデルと同条件 |
| `iterations` | 2000 | early_stoppingで自動決定 |
| `l2_leaf_reg` | 3.0 | L2正則化（CatBoostのデフォルト） |
| `subsample` | 0.8 | Bagging比率 |
| `random_strength` | 1.0 | 分割点のランダム化（過学習抑制） |
| `min_data_in_leaf` | 20 | リーフの最小サンプル数 |

In [ ]:
cb_params = {
    'loss_function': 'RMSE',
    'eval_metric': 'RMSE',
    'depth': 8,
    'learning_rate': 0.02,
    'iterations': 2000,
    'l2_leaf_reg': 3.0,
    'subsample': 0.8,
    'random_strength': 1.0,
    'min_data_in_leaf': 20,
    'random_seed': SEED,
    'verbose': 0,
    'allow_writing_files': False,
}

print('=== CatBoost ハイパーパラメータ ===')
for k, v in cb_params.items():
    print(f'  {k}: {v}')

---
## 4. 時系列バリデーションによる学習

In [ ]:
# === Single Split での学習 ===
X_train = train_df[all_features]
y_train = np.log1p(train_df['visitors'])
X_valid = valid_df[all_features]
y_valid = np.log1p(valid_df['visitors'])

train_pool = Pool(X_train, label=y_train)
valid_pool = Pool(X_valid, label=y_valid)

model = CatBoostRegressor(**cb_params)
model.fit(
    train_pool,
    eval_set=valid_pool,
    early_stopping_rounds=100,
    verbose=200,
)

pred_log = model.predict(X_valid)
pred = np.expm1(pred_log)
score_single = rmsle(valid_df['visitors'], pred)

print(f'\n=== Single Split 結果 ===')
print(f'  Best iteration: {model.best_iteration_}')
print(f'  CatBoost     RMSLE: {score_single:.5f}')
print(f'  LightGBM     RMSLE: {prev_lgbm["score_single"]:.5f}')
print(f'  XGBoost      RMSLE: {prev_xgb["score_single"]:.5f}')
print(f'  RandomForest RMSLE: {prev_rf["score_single"]:.5f}')

In [ ]:
# === 時系列 Cross-Validation（01 EDAで設計した5フォールド） ===
# ※ TimeSeriesSplit ではなく、02で検証済みのval_foldsを使用（スコア比較可能性を担保）
full_df = pd.concat([train_df, valid_df], ignore_index=True).sort_values('visit_date').reset_index(drop=True)

cv_scores = []

for i, fold in enumerate(val_folds, 1):
    val_start = pd.Timestamp(fold['val_start'])
    val_end = pd.Timestamp(fold['val_end'])

    train_mask = full_df['visit_date'] < val_start
    valid_mask = (full_df['visit_date'] >= val_start) & (full_df['visit_date'] <= val_end)

    fold_train = full_df[train_mask]
    fold_valid = full_df[valid_mask]

    if len(fold_train) == 0 or len(fold_valid) == 0:
        print(f'Fold {i}: スキップ（データ不足）')
        continue

    tr_pool = Pool(fold_train[all_features], label=np.log1p(fold_train['visitors']))
    va_pool = Pool(fold_valid[all_features], label=np.log1p(fold_valid['visitors']))

    m = CatBoostRegressor(**cb_params)
    m.fit(tr_pool, eval_set=va_pool, early_stopping_rounds=100, verbose=0)

    p = np.expm1(m.predict(fold_valid[all_features]))
    s = rmsle(fold_valid['visitors'], p)
    cv_scores.append(s)

    print(f'Fold {i}: RMSLE={s:.5f}  '
          f'(train: {fold_train["visit_date"].min().date()}~{fold_train["visit_date"].max().date()}, '
          f'valid: {fold["val_start"]}~{fold["val_end"]}, '
          f'n_train={len(fold_train):,}, n_valid={len(fold_valid):,}, '
          f'best_iter={m.best_iteration_})')

print(f'\n=== CV結果（{len(val_folds)}フォールド） ===')
print(f'  CatBoost     平均 RMSLE: {np.mean(cv_scores):.5f} (+/- {np.std(cv_scores):.5f})')
print(f'  LightGBM     平均 RMSLE: {prev_lgbm["cv_mean"]:.5f} (+/- {prev_lgbm["cv_std"]:.5f})')
print(f'  XGBoost      平均 RMSLE: {prev_xgb["cv_mean"]:.5f} (+/- {prev_xgb["cv_std"]:.5f})')
print(f'  RandomForest 平均 RMSLE: {prev_rf["cv_mean"]:.5f} (+/- {prev_rf["cv_std"]:.5f})')

In [ ]:
# === ルールベースベースライン: 店舗×曜日の過去中央値 ===
# 上位解法の知見: store×DOW中央値はGBDTに匹敵し、アンサンブルで補完効果がある
train_df_with_dow = train_df.copy()
train_df_with_dow['dow'] = train_df_with_dow['visit_date'].dt.dayofweek

store_dow_median = train_df_with_dow.groupby(['air_store_id', 'dow'])['visitors'].median()
global_median = train_df['visitors'].median()

valid_df_with_dow = valid_df.copy()
valid_df_with_dow['dow'] = valid_df_with_dow['visit_date'].dt.dayofweek

baseline_pred = valid_df_with_dow.apply(
    lambda r: store_dow_median.get((r['air_store_id'], r['dow']), global_median), axis=1)
baseline_rmsle = rmsle(valid_df['visitors'], baseline_pred)

print(f'=== ルールベースベースライン ===')
print(f'  store×DOW中央値 RMSLE: {baseline_rmsle:.5f}')
print(f'  CatBoost Single Split: {score_single:.5f}')
print(f'  改善幅: {baseline_rmsle - score_single:.5f}')
print(f'\n  ※ このベースラインは04のアンサンブルで使用する')

---
## 5. 予測結果の分析

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(valid_df['visitors'], pred, alpha=0.1, s=5)
max_val = max(valid_df['visitors'].max(), pred.max())
axes[0].plot([0, max_val], [0, max_val], 'r--', linewidth=1)
axes[0].set_xlabel('実測値')
axes[0].set_ylabel('予測値')
axes[0].set_title('予測 vs 実測（元スケール）')

axes[1].scatter(np.log1p(valid_df['visitors']), pred_log, alpha=0.1, s=5)
max_log = max(np.log1p(valid_df['visitors']).max(), pred_log.max())
axes[1].plot([0, max_log], [0, max_log], 'r--', linewidth=1)
axes[1].set_xlabel('実測値 (log1p)')
axes[1].set_ylabel('予測値 (log1p)')
axes[1].set_title('予測 vs 実測（logスケール）')

residuals = np.log1p(valid_df['visitors'].values) - pred_log
axes[2].hist(residuals, bins=80, color='#C44E52', edgecolor='white', alpha=0.8)
axes[2].axvline(0, color='red', linestyle='--')
axes[2].set_xlabel('残差 (log1p)')
axes[2].set_ylabel('頻度')
axes[2].set_title(f'残差分布 (平均: {residuals.mean():.4f}, 標準偏差: {residuals.std():.4f})')

plt.suptitle('CatBoost 予測結果の分析', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 6. 特徴量重要度の分析

In [ ]:
importance = pd.DataFrame({
    'feature': all_features,
    'importance': model.get_feature_importance()
}).sort_values('importance', ascending=True)

def categorize_feature(name):
    if name.startswith('rolling_') or name.startswith('ewm_'):
        return 'Rolling統計量'
    elif name.startswith('lag_'):
        return 'ラグ特徴量'
    elif name.startswith(('open_ratio_', 'closed_streak', 'days_since_')):
        return '休業パターン'
    elif name.startswith(('is_after_holiday_', 'holiday_count_', 'is_near_special')):
        return '祝日前後'
    elif name.startswith('store_'):
        return '店舗統計量'
    elif name.startswith('genre_'):
        return 'ジャンル'
    else:
        return '時間/店舗属性'

importance['category'] = importance['feature'].apply(categorize_feature)

color_map = {
    'Rolling統計量': '#DD8452', 'ラグ特徴量': '#55A868',
    '店舗統計量': '#4C72B0', 'ジャンル': '#C44E52', '時間/店舗属性': '#8172B3',
    '休業パターン': '#E5AE38', '祝日前後': '#6ACC65'
}

fig, axes = plt.subplots(1, 2, figsize=(18, max(10, len(all_features) * 0.28)))

colors = importance['category'].map(color_map)
axes[0].barh(importance['feature'], importance['importance'], color=colors)
axes[0].set_xlabel('重要度')
axes[0].set_title('CatBoost 個別特徴量の重要度')

cat_imp = importance.groupby('category')['importance'].sum().sort_values()
cat_colors = [color_map.get(c, 'gray') for c in cat_imp.index]
cat_imp.plot(kind='barh', ax=axes[1], color=cat_colors)
axes[1].set_xlabel('重要度合計')
axes[1].set_title('カテゴリ別 重要度合計')

plt.tight_layout()
plt.show()

print('\n=== 上位10特徴量 ===')
for _, row in importance.tail(10).iloc[::-1].iterrows():
    print(f'  [{row["category"]:8s}] {row["feature"]}: {row["importance"]:.2f}')

---
## 7. 全モデル比較とアンサンブル

In [ ]:
# 4モデルの予測値・残差の相関
pred_df = pd.DataFrame({
    'LightGBM': prev_lgbm['valid_pred'],
    'XGBoost': prev_xgb['valid_pred'],
    'RandomForest': prev_rf['valid_pred'],
    'CatBoost': pred,
})

resid_df = pd.DataFrame({
    'LightGBM': prev_lgbm['residuals'],
    'XGBoost': prev_xgb['residuals'],
    'RandomForest': prev_rf['residuals'],
    'CatBoost': residuals,
})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(pred_df.corr(), annot=True, fmt='.4f', cmap='YlOrRd',
            ax=axes[0], square=True, vmin=0.9, vmax=1.0)
axes[0].set_title('予測値の相関')

sns.heatmap(resid_df.corr(), annot=True, fmt='.4f', cmap='YlOrRd',
            ax=axes[1], square=True, vmin=0.5, vmax=1.0)
axes[1].set_title('残差の相関')

plt.tight_layout()
plt.show()

In [ ]:
# 各種アンサンブルの比較
actual = valid_df['visitors'].values

ensembles = {
    'LGB単体':          prev_lgbm['valid_pred'],
    'XGB単体':          prev_xgb['valid_pred'],
    'RF単体':           prev_rf['valid_pred'],
    'CatBoost単体':     pred,
    'LGB+XGB':          (prev_lgbm['valid_pred'] + prev_xgb['valid_pred']) / 2,
    'LGB+XGB+CB':       (prev_lgbm['valid_pred'] + prev_xgb['valid_pred'] + pred) / 3,
    'LGB+XGB+RF':       (prev_lgbm['valid_pred'] + prev_xgb['valid_pred'] + prev_rf['valid_pred']) / 3,
    'LGB+XGB+CB+RF':    (prev_lgbm['valid_pred'] + prev_xgb['valid_pred'] + pred + prev_rf['valid_pred']) / 4,
    'Boosting3平均':    (prev_lgbm['valid_pred'] + prev_xgb['valid_pred'] + pred) / 3,
}

print('=== アンサンブル比較 ===')
print(f'{"組み合わせ":<20s} {"RMSLE":>10s}')
print('-' * 32)
ens_scores = {}
for name, ens_pred in ensembles.items():
    s = rmsle(actual, ens_pred)
    ens_scores[name] = s
    print(f'{name:<20s} {s:>10.5f}')

best_name = min(ens_scores, key=ens_scores.get)
print(f'\n→ 最良の組み合わせ: {best_name} (RMSLE={ens_scores[best_name]:.5f})')

In [ ]:
# スコア比較の可視化
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 単体モデル比較
single_models = ['LGB単体', 'XGB単体', 'RF単体', 'CatBoost単体']
single_scores = [ens_scores[n] for n in single_models]
colors_single = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']
bars = axes[0].bar(single_models, single_scores, color=colors_single)
for bar, val in zip(bars, single_scores):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{val:.4f}', ha='center', fontsize=11, fontweight='bold')
axes[0].set_ylabel('RMSLE')
axes[0].set_title('単体モデルのスコア比較')

# アンサンブル比較
ens_names = [n for n in ens_scores if n not in single_models]
ens_vals = [ens_scores[n] for n in ens_names]
best_single = min(single_scores)
bars2 = axes[1].barh(ens_names, ens_vals, color='#8172B3')
axes[1].axvline(best_single, color='red', linestyle='--', alpha=0.7, label=f'最良単体: {best_single:.4f}')
for bar, val in zip(bars2, ens_vals):
    axes[1].text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=10)
axes[1].set_xlabel('RMSLE')
axes[1].set_title('アンサンブルのスコア比較')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 8. ハイパーパラメータチューニング（Optuna）

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

N_TRIALS = 50

def objective(trial):
    params = {
        'loss_function': 'RMSE',
        'eval_metric': 'RMSE',
        'depth': trial.suggest_int('depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 0.1, 10.0, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'random_strength': trial.suggest_float('random_strength', 0.1, 10.0, log=True),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 5, 50),
        'iterations': 3000,
        'random_seed': SEED,
        'verbose': 0,
        'allow_writing_files': False,
    }

    # val_foldsベースのCV平均で最適化（Single Split過適合を防止）
    fold_scores = []
    for fold in val_folds:
        val_start = pd.Timestamp(fold['val_start'])
        val_end = pd.Timestamp(fold['val_end'])

        train_mask = full_df['visit_date'] < val_start
        valid_mask = (full_df['visit_date'] >= val_start) & (full_df['visit_date'] <= val_end)

        fold_train = full_df[train_mask]
        fold_valid = full_df[valid_mask]

        if len(fold_train) == 0 or len(fold_valid) == 0:
            continue

        tr_pool = Pool(fold_train[all_features], label=np.log1p(fold_train['visitors']))
        va_pool = Pool(fold_valid[all_features], label=np.log1p(fold_valid['visitors']))

        m = CatBoostRegressor(**params)
        m.fit(tr_pool, eval_set=va_pool, early_stopping_rounds=50, verbose=0)

        p = np.expm1(m.predict(fold_valid[all_features]))
        fold_scores.append(rmsle(fold_valid['visitors'], p))

    return np.mean(fold_scores) if fold_scores else float('inf')

study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f'=== Optunaチューニング結果 ({N_TRIALS}試行, {len(val_folds)}フォールドCV) ===')
print(f'  デフォルト CV平均 RMSLE: {np.mean(cv_scores):.5f}')
print(f'  最適化後 CV平均 RMSLE:  {study.best_value:.5f}')
print(f'  改善幅: {np.mean(cv_scores) - study.best_value:+.5f}')
print(f'\n最適パラメータ:')
for k, v in study.best_params.items():
    print(f'  {k}: {v:.6f}' if isinstance(v, float) else f'  {k}: {v}')

# ベストパラメータで再学習（Single Split）
tuned_params = {
    'loss_function': 'RMSE',
    'eval_metric': 'RMSE',
    'iterations': 3000,
    'random_seed': SEED,
    'verbose': 0,
    'allow_writing_files': False,
}
tuned_params.update(study.best_params)

X_train = train_df[all_features]
y_train = np.log1p(train_df['visitors'])
X_valid = valid_df[all_features]
y_valid = np.log1p(valid_df['visitors'])

train_pool = Pool(X_train, label=y_train)
valid_pool = Pool(X_valid, label=y_valid)

tuned_model = CatBoostRegressor(**tuned_params)
tuned_model.fit(train_pool, eval_set=valid_pool, early_stopping_rounds=100, verbose=200)

tuned_pred_log = tuned_model.predict(X_valid)
tuned_pred = np.expm1(tuned_pred_log)
tuned_score = rmsle(valid_df['visitors'], tuned_pred)

print(f'\n=== チューニング後モデル ===')
print(f'  Best iteration: {tuned_model.best_iteration_}')
print(f'  チューニング後 RMSLE: {tuned_score:.5f}')
print(f'  チューニング前 RMSLE: {score_single:.5f}')
print(f'  LightGBM          RMSLE: {prev_lgbm["score_single"]:.5f}')
print(f'  XGBoost            RMSLE: {prev_xgb["score_single"]:.5f}')
print(f'  RandomForest       RMSLE: {prev_rf["score_single"]:.5f}')

---
## 9. まとめ

In [ ]:
print('=' * 60)
print('全モデル設計完了 — 最終比較')
print('=' * 60)

print(f'\n{"モデル":<14s} {"Single RMSLE":>14s} {"CV平均 RMSLE":>14s}')
print('-' * 44)
print(f'{"ベースライン":<14s} {baseline_rmsle:>14.5f} {"---":>14s}')
print(f'{"LightGBM":<14s} {prev_lgbm["score_single"]:>14.5f} {prev_lgbm["cv_mean"]:>14.5f}')
print(f'{"XGBoost":<14s} {prev_xgb["score_single"]:>14.5f} {prev_xgb["cv_mean"]:>14.5f}')
print(f'{"RandomForest":<14s} {prev_rf["score_single"]:>14.5f} {prev_rf["cv_mean"]:>14.5f}')
print(f'{"CatBoost":<14s} {score_single:>14.5f} {np.mean(cv_scores):>14.5f}')
print(f'{"CB Tuned":<14s} {tuned_score:>14.5f} {study.best_value:>14.5f}')

print(f'\n最良アンサンブル: {best_name} → RMSLE={ens_scores[best_name]:.5f}')
print(f'ベースライン対比改善: {baseline_rmsle - ens_scores[best_name]:.5f}')
print(f'\n→ 04_性能比較ノートブックでより詳細なアンサンブル戦略を検討する。')

In [ ]:
# 中間データの保存
results_03_4 = {
    'valid_pred': pred,
    'valid_pred_log': pred_log,
    'valid_actual': valid_df['visitors'].values,
    'score_single': score_single,
    'cv_scores': cv_scores,
    'cv_mean': np.mean(cv_scores),
    'cv_std': np.std(cv_scores),
    'feature_importance': importance,
    'params': cb_params,
    'best_iteration': model.best_iteration_,
    'residuals': residuals,
    'ensemble_scores': ens_scores,
    'best_ensemble': best_name,
    # ルールベースベースライン
    'baseline_rmsle': baseline_rmsle,
    'baseline_pred': baseline_pred.values,
    'store_dow_median': store_dow_median,
    # Optunaチューニング結果
    'tuned_params': tuned_params,
    'tuned_score': tuned_score,
    'tuned_pred': tuned_pred,
    'tuned_pred_log': tuned_pred_log,
    'tuned_best_iteration': tuned_model.best_iteration_,
    'optuna_best_params': study.best_params,
    'optuna_best_value': study.best_value,
}

with open(INTERMEDIATE_DIR / '03-4_catboost_results.pkl', 'wb') as f:
    pickle.dump(results_03_4, f)

# チューニング前モデル保存
model.save_model(str(INTERMEDIATE_DIR / '03-4_catboost_model.cbm'))

# チューニング後モデル保存
tuned_model.save_model(str(INTERMEDIATE_DIR / '03-4_catboost_tuned_model.cbm'))

print('中間データ保存完了:')
print(f'  結果: {INTERMEDIATE_DIR / "03-4_catboost_results.pkl"}')
print(f'  モデル（チューニング前）: {INTERMEDIATE_DIR / "03-4_catboost_model.cbm"}')
print(f'  モデル（チューニング後）: {INTERMEDIATE_DIR / "03-4_catboost_tuned_model.cbm"}')